# LCA Rule Rate Alert — Snowflake Notebooks Demo (Option B)

This notebook demonstrates the **Notebooks + Git** deployment approach for the
LCA Monitoring Reports migration from SAS to Python/Snowpark.

**Architecture:** GitHub Repo → Git-backed Workspace → Notebook Cells → Scheduled Task

Each cell below represents one stage of the daily alert workflow:
1. **Session & Config** — establish connection, load configuration
2. **Freshness Check** — verify source data is current
3. **Alert Loop (N=2..8)** — compute rule-rate metrics for last 7 days
4. **Output & Verification** — write results, display summary
5. **Monitoring** — check task history and run status

In [ ]:
# CELL 1: Session & Configuration
# In Snowflake Notebooks, the session is pre-wired — no credentials needed.
from snowflake.snowpark.context import get_active_session
import datetime

session = get_active_session()

SCHEMA = 'DEMO_DB.LCA_DEMO'
SOURCE_TABLE = f'{SCHEMA}.LCA_TRANSACTIONS_ALL'
OUTPUT_TABLE = f'{SCHEMA}.ALERT_RESULTS'
N_RANGE = range(2, 9)

print(f'Session: {session.get_current_account()}')
print(f'Schema: {SCHEMA}')
print(f'Today: {datetime.date.today()}')
print(f'N range: {list(N_RANGE)}')

In [ ]:
# CELL 2: Freshness Check (Data Freshness Guard)
freshness_df = session.sql(f'''
    SELECT MAX(submission_date) AS max_sub_date,
           CURRENT_DATE() AS today,
           DATEDIFF('day', MAX(submission_date), CURRENT_DATE()) AS staleness_days
    FROM {SOURCE_TABLE}
''')
freshness_df.show()

row = freshness_df.collect()[0]
max_sub_date = row['MAX_SUB_DATE']
if hasattr(max_sub_date, 'date'):
    max_sub_date = max_sub_date.date()

today = datetime.date.today()
lst_dt_t = max_sub_date - datetime.timedelta(days=1) if max_sub_date == today else max_sub_date
staleness = (today - lst_dt_t).days
assert staleness <= 10, f'Data is {staleness} days stale!'

print(f'Freshness check PASSED: lst_dt_t={lst_dt_t}, staleness={staleness} day(s)')

In [ ]:
# CELL 3: Alert Loop (N=2..8)
session.sql(f'TRUNCATE TABLE {OUTPUT_TABLE}').collect()
results_inserted = 0

for n in N_RANGE:
    report_date = today - datetime.timedelta(days=n)
    if lst_dt_t < (today - datetime.timedelta(days=n)):
        print(f'  N={n}: SKIPPED (data not fresh enough)')
        continue
    rows = session.sql(f'''
        SELECT submission_date AS report_date, rule_name,
               AVG(rule_rate) AS avg_rule_rate, SUM(loan_count) AS total_loans
        FROM {SOURCE_TABLE}
        WHERE submission_date = '{report_date}'
        GROUP BY submission_date, rule_name
    ''').collect()
    if not rows:
        print(f'  N={n}: SKIPPED (no data for {report_date})')
        continue
    for row in rows:
        session.sql(f'''
            INSERT INTO {OUTPUT_TABLE} (report_date, n_value, rule_name, avg_rule_rate, total_loans)
            VALUES ('{row["REPORT_DATE"]}', {n}, '{row["RULE_NAME"]}', {row["AVG_RULE_RATE"]}, {row["TOTAL_LOANS"]})
        ''').collect()
        results_inserted += 1
    print(f'  N={n}: Report_Date={report_date}, {len(rows)} rule(s) inserted')

print(f'\nAlert loop complete: {results_inserted} rows inserted into ALERT_RESULTS')

In [ ]:
# CELL 4: Output & Verification
results_df = session.table(OUTPUT_TABLE).order_by('N_VALUE')
print(f'ALERT_RESULTS: {results_df.count()} rows')
print(f'Period: {today - datetime.timedelta(days=2)} to {today - datetime.timedelta(days=8)}')
results_df.show()

In [ ]:
-- CELL 5: Monitoring (SQL Cell)
SELECT report_date, n_value, rule_name,
       ROUND(avg_rule_rate, 4) AS avg_rule_rate,
       total_loans, generated_at
FROM DEMO_DB.LCA_DEMO.ALERT_RESULTS
ORDER BY n_value

## Summary: Notebook vs SnowGit Approach

| Aspect | Option A (SnowGit) | Option B (Notebooks) |
|--------|-------------------|---------------------|
| Session | `Session.builder.configs(...)` | `get_active_session()` — pre-wired |
| Code format | Plain `.py` files | Notebook cells (`.ipynb`) |
| Scheduling | Snowflake Task (manual setup) | Built-in scheduler (click button) |
| Git sync | One-way pull (FETCH) | Bidirectional push & pull |
| Debugging | Run locally, check logs | Cell-by-cell with visual output |
| Error handling | `sys.exit(1)` + Task History | `assert` + cell error display |

### To Schedule This Notebook
1. Click **Schedule** in the notebook toolbar
2. Set: Daily at 6:00 AM ET
3. Snowflake creates a Task behind the scenes
4. View run history in the **Runs** tab